# LightGBM GPU Demo Notebook

この notebook は **CUDA / GPU 前提** の LightGBM 学習デモです。

## 先にやること

Jupyter 依存の追加後は、LightGBM が CPU 版 wheel に戻ることがあります。
そのため、**この notebook を開く前に** 一度以下を実行してください。

```bash
uv sync
./scripts/build_lightgbm_cuda.sh
jupyter lab
```

notebook 内の学習セルは `device_type="cuda"` を使います。


In [ ]:
from pathlib import Path
import json

import lightgbm as lgb
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split


In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print('Root:', ROOT)
print('LightGBM version:', lgb.__version__)


In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_classes=2,
    weights=[0.55, 0.45],
    class_sep=1.0,
    random_state=42,
)

columns = [f'feature_{i:02d}' for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=columns)
df['target'] = y
df.head()


In [ ]:
dataset_path = DATA_DIR / 'dummy_tabular_data.csv'
df.to_csv(dataset_path, index=False)
print('Saved:', dataset_path)


In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)


In [ ]:
model = lgb.LGBMClassifier(
    objective='binary',
    device_type='cuda',
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    max_bin=255,
    random_state=42,
)

model.fit(X_train, y_train)


In [ ]:
pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    'accuracy': round(float(accuracy_score(y_test, pred)), 4),
    'roc_auc': round(float(roc_auc_score(y_test, pred_proba)), 4),
    'device_type': 'cuda',
}
metrics


In [ ]:
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
importance_df


In [ ]:
metrics_path = OUTPUT_DIR / 'metrics.json'
importance_path = OUTPUT_DIR / 'feature_importance.csv'
report_path = OUTPUT_DIR / 'classification_report.txt'

metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
importance_df.to_csv(importance_path, index=False)
report_path.write_text(classification_report(y_test, pred), encoding='utf-8')

print(metrics_path)
print(importance_path)
print(report_path)


In [ ]:
print(classification_report(y_test, pred))
